<a href="https://colab.research.google.com/github/vrishinm/2026-Data3001-XIE-G2/blob/Zack/Validation_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# DATA3001 — Validation of the 7-day transition matrix

Drifter-level 60/20/20 split, baselines, multiclass Brier score and top-3 hit rate (README, RQ3 "Predictive performance").

**Needs:** `pairs7.csv` from `Proposal_code.ipynb` (Section 6), found in `gdp_out/` or next to the notebook (`PAIRS_PATH`), the group's drifter ID table `drifter_ids.csv` (1,126 drifters with pairs; columns `traj, gdp_id, n_pairs, ...`) and the GDP hourly data via CloudDrift. It does not rebuild the pairs. If the pairs are refined later and a drifter appears that is not in the ID table, the notebook stops and tells you, rather than quietly splitting it differently.

**Writes:** `gdp_out/drifter_split.csv` (the ID table plus a `split` column; commit it), `gdp_out/validation_val.csv`, `gdp_out/validation_test.csv`.

**Discipline:** settings are chosen on the validation drifters only. The test drifters are scored once, at the end (Section 6 is locked until `TEST_UNLOCKED = True`).

| Model | Prediction for a pair starting in cell *i* |
|---|---|
| **Transition matrix P** | row *i* of P, estimated on the training drifters (counts / row total, as in the README) |
| **Persistence** | the drifter is still in cell *i* after 7 days (probability 1) |
| **Overall frequency** | the same distribution for every pair: how often each destination (cells and the four exits) occurs among all training pairs |
| **Mean current** | start at the centre of cell *i*, move by the training drifters' mean 7-day displacement from that cell, and read off the cell (or exit side) it lands in (probability 1) |

**Scores.** Multiclass Brier score = mean over test pairs of Σ<sub>k</sub> (p<sub>k</sub> − 1[k = observed destination])², over the 238 states (234 cells + 4 exits). Range 0 to 2, lower is better; a confident wrong guess costs 2. Top-3 hit rate = share of pairs whose observed destination is among the three most probable states (a state only counts as a guess if its probability is above 0, so persistence and mean current, which make a single guess, have top-3 = top-1). As a second view that speaks to RQ1 directly, every model is also scored on 5 coarse outcomes (stayed in R, exited west / east / south / north).

In [42]:
!pip install -q clouddrift

### 0.1. Upload .csv files

In [43]:
from google.colab import files

uploaded = files.upload()

Saving pairs7.csv to pairs7 (1).csv


In [44]:
from google.colab import files

uploaded = files.upload()

Saving drifter_ids.csv to drifter_ids (1).csv


### 0.2. Setup

In [45]:
import os, json, time
import numpy as np
import pandas as pd

# Region, grid and state numbering: identical to Proposal_code.ipynb (Sections 0 and 6)
west, east, south, north = 10, 45, -45, -20
GRID = 2
H = 7 * 24                          # 7 days on the hourly grid
DAY = np.int64(86_400 * 10**9)      # one day in nanoseconds
OUT = "gdp_out"
os.makedirs(OUT, exist_ok=True)

nx = int(np.ceil((east - west) / GRID))
ny = int(np.ceil((north - south) / GRID))
n_cells = nx * ny                   # 18 x 13 = 234 cells
EXIT = {"west": n_cells, "east": n_cells + 1, "south": n_cells + 2, "north": n_cells + 3}
n_states = n_cells + 4              # cells 0..233, then 4 absorbing exit states
OUTCOMES = ["stayed", "west", "east", "south", "north"]

# Settings
SPLIT_SEED = 3001
SPLIT_FRACS = (0.60, 0.20, 0.20)    # train / val / test, by whole drifter
PAIRS_PATH = next((p for p in (os.path.join(OUT, "pairs7.csv"), "pairs7.csv") if os.path.exists(p)),
                  os.path.join(OUT, "pairs7.csv"))   # gdp_out/pairs7.csv, or pairs7.csv next to the notebook
ID_TABLE_PATH = "drifter_ids.csv"   # the group's ID table: one row per drifter (gdp_id), from the 7-day pairs
SPLIT_PATH = os.path.join(OUT, "drifter_split.csv")   # same table + a train/val/test column
MIN_DRIFTERS = 10                   # a start cell is "supported" if >= 10 distinct TRAINING drifters start there (README success criterion a)
N_BOOT = 2000                       # drifter-level bootstrap draws for the confidence intervals
TEST_UNLOCKED = False               # flip to True only for the final, one-off test evaluation (Section 6)

def in_box(lo, la, x0, x1, y0, y1):
    return (lo >= x0) & (lo <= x1) & (la >= y0) & (la <= y1)

def cell_of(lo, la, x0, x1, y0, y1, size):
    """Cell number inside a box; np.clip keeps points exactly on the east/north edge in the last column/row."""
    nc, nr = int(np.ceil((x1 - x0) / size)), int(np.ceil((y1 - y0) / size))
    c = np.clip(np.floor((lo - x0) / size).astype(int), 0, nc - 1)
    r = np.clip(np.floor((la - y0) / size).astype(int), 0, nr - 1)
    return r * nc + c, nc, nr

## 1. Load the pairs and the positions they need

The mean-current baseline needs each pair's position 7 days after the start (also when the drifter has left R, because `pairs7` follows whole tracks), so the hourly positions are loaded once, the few columns we need are attached to the pairs, and the big arrays are freed.

In [46]:
from clouddrift.datasets import gdp1h

def load(da, name, tries=5):
    """.values with a few retries: the public S3 store sometimes drops mid-transfer."""
    for i in range(tries):
        try:
            return da.values
        except Exception as e:
            if i == tries - 1:
                raise
            print(f"  {name} failed ({e!r}), retrying ...")
            time.sleep(3 * (i + 1))

t0 = time.time()
if "ds" not in globals():
    ds = gdp1h()
lon = load(ds["lon"], "lon")
lat = load(ds["lat"], "lat")
t_ns = load(ds["time"], "time").astype("datetime64[ns]").astype(np.int64)
rowsize = load(ds["rowsize"], "rowsize").astype(np.int64)
drifter_id_all = load(ds["id"], "id").astype(np.int64)      # GDP drifter ID, one per trajectory
bounds = np.cumsum(rowsize)
assert bounds[-1] == len(lon)
print(f"{len(lon):,} hourly records, {len(rowsize):,} drifters, loaded in {time.time() - t0:.0f}s")

# ---- pairs from Proposal_code.ipynb
pairs = pd.read_csv(PAIRS_PATH, parse_dates=["time"])
print(f"read {PAIRS_PATH}")
assert {"origin_cell", "dest", "traj", "time", "obs_row"} <= set(pairs.columns)
r0, tj = pairs.obs_row.to_numpy(), pairs.traj.to_numpy()

# the CSV must belong to the data we just loaded
assert in_box(lon[r0], lat[r0], west, east, south, north).all(), f"{PAIRS_PATH} does not match this dataset (start not in R)"
assert (t_ns[r0] == pairs.time.to_numpy().astype("datetime64[ns]").astype(np.int64)).all(), f"{PAIRS_PATH} times do not match this dataset"
assert (np.searchsorted(bounds, r0, side="right") == tj).all(), f"{PAIRS_PATH} traj does not match obs_row"

# start position, 7-day displacement (degrees) and drifter ID
r7 = r0 + H
has7 = r7 < bounds[tj]                                   # still inside the same drifter's rows
r7s = np.where(has7, r7, r0)
has7 &= (t_ns[r7s] - t_ns[r0]) == 7 * DAY                # real elapsed time, not just the row offset
has7 &= np.isfinite(lon[r7s]) & np.isfinite(lat[r7s])
pairs["lon0"], pairs["lat0"] = lon[r0], lat[r0]
pairs["dlon"] = np.where(has7, lon[r7s] - lon[r0], np.nan)
pairs["dlat"] = np.where(has7, lat[r7s] - lat[r0], np.nan)
pairs["drifter_id"] = drifter_id_all[tj]
pairs["outcome"] = np.where(pairs.dest < n_cells, 0, pairs.dest - n_cells + 1)   # 0 stayed, 1 west, 2 east, 3 south, 4 north
pairs["year"] = pairs.time.dt.year

print(f"pairs: {len(pairs):,} from {pairs.drifter_id.nunique():,} drifters | "
      f"with a day-7 position (needed for mean current): {has7.mean():.1%}")
del lon, lat, t_ns, r0, r7, r7s, tj, has7

197,214,787 hourly records, 19,396 drifters, loaded in 4s
read gdp_out/pairs7.csv
pairs: 154,139 from 1,126 drifters | with a day-7 position (needed for mean current): 99.7%


## 2. The shared drifter split (60 / 20 / 20)

The drifters are the rows of `drifter_ids.csv`, keyed on `gdp_id`. The ID table has no split column yet, so this section adds one: a seeded random draw by whole drifter. The result is saved as `gdp_out/drifter_split.csv` and is the source of truth: if it exists it is loaded and checked, never silently regenerated, so everyone who pulls it from GitHub uses exactly the same train / validation / test drifters. (The draw uses numpy's legacy `RandomState`, whose stream is frozen across versions, but the CSV is what counts.)

In [47]:
def make_split(drifter_ids, fracs=SPLIT_FRACS, seed=SPLIT_SEED):
    """Random split by whole drifter. IDs are sorted first, so the result does not depend on the row order of the file."""
    ids = np.sort(np.unique(np.asarray(drifter_ids, dtype=np.int64)))
    n = len(ids)
    n_tr, n_va = int(round(fracs[0] * n)), int(round(fracs[1] * n))
    perm = np.random.RandomState(seed).permutation(n)
    split = np.empty(n, dtype=object)
    split[perm[:n_tr]] = "train"
    split[perm[n_tr:n_tr + n_va]] = "val"
    split[perm[n_tr + n_va:]] = "test"
    return pd.DataFrame({"gdp_id": ids, "split": split})

# ---- the group's ID table
id_tab = pd.read_csv(ID_TABLE_PATH)
assert {"traj", "gdp_id"} <= set(id_tab.columns), f"{ID_TABLE_PATH} needs columns traj and gdp_id"
assert id_tab.gdp_id.is_unique and id_tab.traj.is_unique, f"{ID_TABLE_PATH} has duplicate drifters"
assert (drifter_id_all[id_tab.traj.to_numpy()] == id_tab.gdp_id.to_numpy()).all(), \
    "gdp_id does not match traj in this dataset: the ID table was made from a different data release"
print(f"ID table: {len(id_tab):,} drifters" + (f", {id_tab.n_pairs.sum():,} pairs" if "n_pairs" in id_tab else ""))

def load_or_make_split(id_tab, path):
    if os.path.exists(path):
        tab = pd.read_csv(path)
        assert tab.gdp_id.is_unique and set(tab.split) <= {"train", "val", "test"}, f"{path} is malformed"
        missing = np.setdiff1d(id_tab.gdp_id.to_numpy(), tab.gdp_id.to_numpy())
        if len(missing):
            raise ValueError(f"{path} has no split for {len(missing)} drifters in the ID table (e.g. {missing[:5]}). "
                             "The ID table has changed: agree on a new split with the group instead of overwriting.")
        extra = np.setdiff1d(tab.gdp_id.to_numpy(), id_tab.gdp_id.to_numpy())
        if len(extra):
            print(f"note: {len(extra)} IDs in {path} are not in the ID table (ignored)")
        print(f"loaded the shared split from {path}")
        return tab
    tab = id_tab.merge(make_split(id_tab.gdp_id), on="gdp_id", how="left")
    tab.to_csv(path, index=False)
    print(f"created {path}: share this file with the group before anyone fits or scores anything")
    return tab

split_tab = load_or_make_split(id_tab, SPLIT_PATH)
split_of = dict(zip(split_tab.gdp_id.to_numpy(), split_tab.split.to_numpy()))
pairs["split"] = pairs.drifter_id.map(split_of)
if pairs.split.isna().any():
    raise ValueError(f"{pairs.loc[pairs.split.isna(), 'drifter_id'].nunique()} drifters in {PAIRS_PATH} are not in {ID_TABLE_PATH}: "
                     "the pairs have changed since the ID table was made; update the table with the group.")
if "n_pairs" in id_tab:
    chk = pairs.groupby("traj").size().reindex(id_tab.traj.to_numpy(), fill_value=0).to_numpy()
    n_diff = int((chk != id_tab.n_pairs.to_numpy()).sum())
    print("pairs per drifter match the ID table" if n_diff == 0 else
          f"note: pairs per drifter differ from the ID table for {n_diff} drifters (pairs refined since?); the split is still valid")

# leakage guard: no drifter in more than one split
assert (pairs.groupby("drifter_id").split.nunique() == 1).all(), "a drifter appears in more than one split"

train = pairs[pairs.split == "train"]
val   = pairs[pairs.split == "val"]
test  = pairs[pairs.split == "test"]

# does the split look balanced? (drifters are split 60/20/20; pairs and outcomes only roughly so)
g = pairs.groupby("split")
rep = pd.DataFrame({"drifters_in_table": split_tab.groupby("split").size(),
                    "drifters_with_pairs": g.drifter_id.nunique(), "pairs": g.size()}).loc[["train", "val", "test"]]
rep["pair_share"] = (rep.pairs / rep.pairs.sum()).round(3)
for k, name in enumerate(OUTCOMES):
    rep[f"share_{name}"] = g.outcome.apply(lambda s, k=k: (s == k).mean()).round(3)
rep["median_year"] = g.year.median()
print(rep.to_string())

ID table: 1,126 drifters, 154,139 pairs
loaded the shared split from gdp_out/drifter_split.csv
pairs per drifter match the ID table
       drifters_in_table  drifters_with_pairs  pairs  pair_share  share_stayed  share_west  share_east  share_south  share_north  median_year
split                                                                                                                                        
train                676                  676  91153       0.591         0.928       0.023       0.033        0.012        0.004       2014.0
val                  225                  225  31622       0.205         0.924       0.026       0.039        0.008        0.003       2012.0
test                 225                  225  31364       0.203         0.928       0.021       0.034        0.015        0.003       2012.0


## 3. Models

Every model is stored in the same form: an `n_states × n_states` row-stochastic matrix `M`, where row *i* is the predicted distribution of the destination for a pair starting in cell *i* (exit-state rows are absorbing). That lets one scoring routine handle all four models, and `P` can later be raised to a power for the 28-day check (`np.linalg.matrix_power(P, 4)`).

Cells that fewer than `MIN_DRIFTERS` training drifters start in are *unsupported*. They are excluded from scoring for **all** models alike (the report below says how many pairs that removes), until the poorly supported cells are merged with neighbours as planned in the README. If you merge cells, remap `origin_cell` and `dest` in `pairs7.csv` to the merged IDs (keeping IDs inside `0..n_cells-1`) and everything here still works.

In [48]:
def fit_P(train, alpha=0.0, min_drifters=MIN_DRIFTERS):
    """7-day transition matrix from training pairs: counts / row total (README definition) when alpha = 0.

    alpha > 0 shrinks each row towards the overall training destination frequencies q:
    (counts + alpha * q) / (row total + alpha). It is a setting to choose on the validation drifters.
    Start cells with no training pairs at all get q so that P stays a valid stochastic matrix.
    Returns P, the boolean `supported` mask over cells, and distinct training drifters per start cell.
    """
    o, y = train.origin_cell.to_numpy(), train.dest.to_numpy()
    counts = np.bincount(o * n_states + y, minlength=n_states * n_states).reshape(n_states, n_states).astype(float)
    n_drifters = (train.groupby("origin_cell").drifter_id.nunique()
                       .reindex(range(n_cells), fill_value=0).to_numpy())
    supported = n_drifters >= min_drifters
    q = counts.sum(0) / counts.sum()
    rowtot = counts[:n_cells].sum(1, keepdims=True)
    P = np.eye(n_states)                                    # exit states: absorbing
    P[:n_cells] = (counts[:n_cells] + alpha * q) / np.where(rowtot + alpha > 0, rowtot + alpha, 1.0)
    P[np.nonzero(rowtot[:, 0] == 0)[0], :] = q
    return P, supported, n_drifters

def persistence_matrix():
    """Stays put: still in the starting cell after 7 days."""
    return np.eye(n_states)

def overall_frequency_matrix(train):
    """Same distribution for every start cell: how often each destination occurs among all training pairs."""
    q = np.bincount(train.dest.to_numpy(), minlength=n_states).astype(float)
    return np.tile(q / q.sum(), (n_states, 1))

def classify_points(lo, la):
    """Cell or exit state for arbitrary end points; same rule as the pairs in Proposal_code.ipynb Section 6."""
    cell = cell_of(lo, la, west, east, south, north, GRID)[0]
    exit_state = np.where(lo < west, EXIT["west"],
                 np.where(lo > east, EXIT["east"],
                 np.where(la < south, EXIT["south"], EXIT["north"])))
    return np.where(in_box(lo, la, west, east, south, north), cell, exit_state)

def mean_current_matrix(train):
    """Advect the centre of each start cell by the mean 7-day displacement of the training drifters that started
    there; the prediction is the cell (or exit side) that point lands in. Needs only the cell centre, so it sees
    the same information as P. A cell with no training displacement falls back to persistence."""
    cells = np.arange(n_cells)
    d = train.dropna(subset=["dlon", "dlat"]).groupby("origin_cell").agg(
        dlon=("dlon", "mean"), dlat=("dlat", "mean"), n_pairs=("dlon", "size"))
    d = d.reindex(cells)
    clon = np.minimum(west + GRID * (cells % nx) + GRID / 2, east)
    clat = np.minimum(south + GRID * (cells // nx) + GRID / 2, north)
    has = d.dlon.notna().to_numpy()
    land = classify_points(clon + d.dlon.fillna(0).to_numpy(), clat + d.dlat.fillna(0).to_numpy())
    pred = np.where(has, land, cells)
    M = np.eye(n_states)
    M[cells, :] = 0
    M[cells, pred] = 1
    return M, d

P, supported, n_drifters_per_cell = fit_P(train)
M_pers = persistence_matrix()
M_freq = overall_frequency_matrix(train)
M_mc, mc_table = mean_current_matrix(train)

models = {"Transition matrix P": P, "Persistence": M_pers, "Overall frequency": M_freq, "Mean current": M_mc}
for name, M in models.items():
    assert np.allclose(M.sum(1), 1) and (M >= 0).all(), f"{name}: rows must be probability distributions"

occupied = (train.groupby("origin_cell").size().reindex(range(n_cells), fill_value=0) > 0).sum()
print(f"Training drifters: {train.drifter_id.nunique():,} | pairs: {len(train):,}")
print(f"Start cells with any training pair: {occupied} | supported (>= {MIN_DRIFTERS} distinct training drifters): {supported.sum()}")

Training drifters: 676 | pairs: 91,153
Start cells with any training pair: 175 | supported (>= 10 distinct training drifters): 164


## 4. Scoring

Scores are computed per pair. Because pairs from the same drifter are strongly dependent (one start per day), confidence intervals come from a **drifter-level bootstrap** (resample whole drifters, not pairs), and model-vs-baseline differences are paired on the same resampled drifters.

In [49]:
def brier_pairs(M, origin, y):
    """Multiclass Brier per pair: sum_k (p_k - 1[k=y])^2 = sum_k p_k^2 - 2 p_y + 1. M is (n_rows, K), y in 0..K-1."""
    rowsq = (M ** 2).sum(1)
    return rowsq[origin] - 2 * M[origin, y] + 1

def topk_hits(M, origin, y, k=3):
    """1 if the observed destination is among the k most probable states of the start cell's row.
    A state only counts as a guess if its probability is > 0 (so one-hot models make a single guess)."""
    order = np.argsort(-M, axis=1, kind="stable")[:, :k]
    valid = np.take_along_axis(M, order, axis=1) > 0
    return ((order[origin] == y[:, None]) & valid[origin]).any(1)

def collapse5(M):
    """Merge the 234 cells into 'stayed in R' and keep the four exits: 5 coarse outcomes."""
    return np.column_stack([M[:, :n_cells].sum(1)] + [M[:, EXIT[s]] for s in ("west", "east", "south", "north")])

def cluster_bootstrap(arrays, cluster, n_boot=N_BOOT, seed=0):
    """Drifter-level bootstrap of the mean of each per-pair array. All arrays share the same resamples (paired)."""
    codes, uniq = pd.factorize(cluster)
    k = len(uniq)
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, k, size=(n_boot, k))
    den = np.bincount(codes).astype(float)[idx].sum(1)
    return {key: np.bincount(codes, weights=v)[idx].sum(1) / den for key, v in arrays.items()}

def evaluate(models, eval_pairs, supported, focus="Transition matrix P", ref="Overall frequency", n_boot=N_BOOT):
    """Score every model on the same pairs: those whose start cell is supported by the training drifters."""
    keep = supported[eval_pairs.origin_cell.to_numpy()]
    sub = eval_pairs[keep]
    o, y, y5 = sub.origin_cell.to_numpy(), sub.dest.to_numpy(), sub.outcome.to_numpy()
    info = dict(pairs_total=len(eval_pairs), pairs_scored=len(sub), drifters_scored=sub.drifter_id.nunique(),
                share_pairs_unsupported=1 - len(sub) / len(eval_pairs))
    per = {}
    for name, M in models.items():
        per[name] = {"bs": brier_pairs(M, o, y), "bs5": brier_pairs(collapse5(M), o, y5),
                     "top1": topk_hits(M, o, y, 1).astype(float), "top3": topk_hits(M, o, y, 3).astype(float)}
    boot = cluster_bootstrap({f"{n}|{m}": v for n, d in per.items() for m, v in d.items()},
                             sub.drifter_id.to_numpy(), n_boot)
    ci = lambda x: f"[{np.percentile(x, 2.5):.3f}, {np.percentile(x, 97.5):.3f}]"
    rows, diffs = [], []
    for name in models:
        bs = per[name]["bs"].mean()
        rows.append({"model": name, "Brier": bs, "Brier 95% CI": ci(boot[f"{name}|bs"]),
                     "skill vs " + ref.lower(): 1 - bs / per[ref]["bs"].mean(),
                     "Brier (5 outcomes)": per[name]["bs5"].mean(),
                     "top-1": per[name]["top1"].mean(), "top-3": per[name]["top3"].mean(),
                     "top-3 95% CI": ci(boot[f"{name}|top3"])})
        if name != focus:
            db = boot[f"{name}|bs"] - boot[f"{focus}|bs"]          # > 0: the matrix has the lower Brier score
            d5 = boot[f"{name}|bs5"] - boot[f"{focus}|bs5"]
            dt = boot[f"{focus}|top3"] - boot[f"{name}|top3"]      # > 0: the matrix has the higher top-3 hit rate
            diffs.append({"baseline": name,
                          "Brier: baseline - P": per[name]["bs"].mean() - per[focus]["bs"].mean(), "95% CI": ci(db),
                          "Brier 5 outcomes: baseline - P": per[name]["bs5"].mean() - per[focus]["bs5"].mean(), "95% CI ": ci(d5),
                          "top-3: P - baseline": per[focus]["top3"].mean() - per[name]["top3"].mean(), "95% CI  ": ci(dt)})
    return pd.DataFrame(rows).set_index("model"), pd.DataFrame(diffs).set_index("baseline"), info

def show(summary, diffs, info, title):
    print(f"{title}: {info['pairs_scored']:,} pairs from {info['drifters_scored']:,} drifters scored "
          f"({info['share_pairs_unsupported']:.1%} of pairs dropped: start cell not supported by the training drifters)")
    print("\nScores (lower Brier is better, higher top-k is better)")
    print(summary.round(3).to_string())
    print("\nPaired differences (positive = the matrix is better; drifter-level bootstrap)")
    print(diffs.round(3).to_string())

## 5. Validation drifters: baselines and settings

The first table is the comparison the README asks for, on the validation drifters. The second is the one setting we can tune: how strongly to shrink sparse rows of P towards the overall frequencies (`alpha = 0` is the plain README matrix). Choose `ALPHA` from this table (or keep 0), write it down, then do not look at the test drifters until Section 6.

In [50]:
summ_val, diff_val, info_val = evaluate(models, val, supported)
show(summ_val, diff_val, info_val, "VALIDATION")
summ_val.to_csv(os.path.join(OUT, "validation_val.csv"))
diff_val.to_csv(os.path.join(OUT, "validation_val_paired_differences.csv"))

VALIDATION: 31,321 pairs from 224 drifters scored (1.0% of pairs dropped: start cell not supported by the training drifters)

Scores (lower Brier is better, higher top-k is better)
                     Brier    Brier 95% CI  skill vs overall frequency  Brier (5 outcomes)  top-1  top-3    top-3 95% CI
model                                                                                                                   
Transition matrix P  0.809  [0.799, 0.819]                       0.184               0.096  0.309  0.632  [0.613, 0.651]
Persistence          1.411  [1.380, 1.441]                      -0.422               0.152  0.295  0.295  [0.279, 0.310]
Overall frequency    0.992  [0.992, 0.992]                       0.000               0.144  0.039  0.073  [0.064, 0.084]
Mean current         1.415  [1.383, 1.446]                      -0.426               0.134  0.293  0.293  [0.277, 0.308]

Paired differences (positive = the matrix is better; drifter-level bootstrap)
              

In [51]:
# Shrinkage setting, chosen on the validation drifters only
sub_v = val[supported[val.origin_cell.to_numpy()]]
o_v, y_v, y5_v = sub_v.origin_cell.to_numpy(), sub_v.dest.to_numpy(), sub_v.outcome.to_numpy()
rows = []
for a in [0, 0.5, 1, 2, 5, 10, 20, 50]:
    Pa, _, _ = fit_P(train, alpha=a)
    rows.append({"alpha": a, "Brier": brier_pairs(Pa, o_v, y_v).mean(),
                 "Brier (5 outcomes)": brier_pairs(collapse5(Pa), o_v, y5_v).mean(),
                 "top-3": topk_hits(Pa, o_v, y_v, 3).mean()})
alpha_grid = pd.DataFrame(rows).set_index("alpha")
print(alpha_grid.round(4).to_string())
print("\nlowest validation Brier at alpha =", alpha_grid.Brier.idxmin())

        Brier  Brier (5 outcomes)   top-3
alpha                                    
0.0    0.8092              0.0958  0.6318
0.5    0.8092              0.0958  0.6324
1.0    0.8092              0.0958  0.6324
2.0    0.8092              0.0958  0.6324
5.0    0.8091              0.0958  0.6324
10.0   0.8090              0.0959  0.6324
20.0   0.8089              0.0960  0.6324
50.0   0.8096              0.0966  0.6323

lowest validation Brier at alpha = 20.0


## 6. Test drifters: final evaluation (run once)

Set `ALPHA` from Section 5 and `TEST_UNLOCKED = True` in Section 0, then run this cell a single time. The matrix is fitted on the training drifters only, as in the README.

In [52]:
ALPHA = 0.0          # from the validation table in Section 5

assert TEST_UNLOCKED, "Test drifters are locked: finish choosing settings on the validation drifters, then set TEST_UNLOCKED = True in Section 0."
P_final, supported_final, _ = fit_P(train, alpha=ALPHA)
models_test = {**models, "Transition matrix P": P_final}
summ_test, diff_test, info_test = evaluate(models_test, test, supported_final)
show(summ_test, diff_test, info_test, f"TEST (alpha = {ALPHA})")
summ_test.to_csv(os.path.join(OUT, "validation_test.csv"))
diff_test.to_csv(os.path.join(OUT, "validation_test_paired_differences.csv"))

AssertionError: Test drifters are locked: finish choosing settings on the validation drifters, then set TEST_UNLOCKED = True in Section 0.

**Reading the results.** The Brier score punishes confident wrong guesses (cost 2), so the one-guess baselines (persistence, mean current) are scored harshly relative to a probabilistic P; the top-3 hit rate and the 5-outcome score give a fairer side-by-side. The split is random by drifter, so test drifters share years and regions with training drifters: this measures how well P predicts drifters it has not seen, not how well it forecasts a future period.

**Next:** `P⁴ = np.linalg.matrix_power(P, 4)` against observed 28-day outcomes on these same train/val/test drifters (same first-exit rule), then the practical example.